# Part 5: Build an agent over the knowledge base

Part 5 at Caldova begins when the supply-chain team wants an agent to reason over evidence without asking the knowledge base to synthesize an answer. You will create an extractive knowledge base from the Part 1 knowledge sources, connect Microsoft Agent Framework to its MCP endpoint, then let the agent work with the retrieved evidence.

**🎯 Mission**
- Create a model-free knowledge base configured for extractive output
- Connect Microsoft Agent Framework to the knowledge base through its MCP endpoint
- Run an operational query and inspect the agent's tool activity

## Where you are in the architecture

This exercise returns to the three Part 1 sources. It does not connect the combined Part 4 KB to an agent. The new extractive KB has no attached model. Your local Agent Framework process calls its MCP endpoint and uses Azure OpenAI to synthesize the answer. No hosted Foundry agent is deployed.

[![Part 5 before: planned local agent, MCP tool client, automatically exposed KB endpoint and extractive KB, with existing sources and model faded.](../../img/architecture/part5-before.png)](../../img/architecture/part5-before.png)

**Architecture in brief:** A local Python agent calls an extractive three-source KB through knowledge_base_retrieve. The KB has no model. The agent calls Azure OpenAI for the final answer. No hosted agent is created.

**Read the map:** the notebook runs the local Agent Framework agent and displays its answer. Dashed orange marks the planned agent, MCP tool client and extractive KB. Search automatically exposes the KB's MCP endpoint, not a separate deployed server. Faded blue cards are the three existing Part 1 sources and the chat model. Faint arrows show planned calls. MCP uses the Search key and the model client uses the Azure OpenAI key. Fabric, Work IQ, Entra sign-in and the earlier upload path are omitted because this runtime does not use them.

The lab has separate KB variants. Part 1 branches into Part 2 with Fabric and Part 3 with Work IQ. Part 4 combines them. Part 5 reuses the Part 1 sources.

See the [visual field guide](../../docs/visual-guide.md) for portal navigation, expected observations, and the screenshot reference gallery.

## Step 1: Set up the agent environment

Load the Azure OpenAI model and Foundry IQ settings from the project `.env` file. These settings create the extractive knowledge base and connect it to the Agent Framework model.

In [ ]:
import os

from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework_openai import OpenAIChatClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import AsyncOpenAI

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
MODEL_DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]

KNOWLEDGE_BASE_NAME = "caldova-supply-chain-minimal-knowledge-base"
AZURE_SEARCH_API_VERSION = "2026-08-01-preview"

print("Agent environment loaded")

In [ ]:
from lab_visuals import (
    show_activity,
    show_architecture,
    show_file_ingestion,
    show_portal_links,
    show_references,
    verify_knowledge_base,
    verify_sources,
)

show_portal_links(
    AZURE_SEARCH_SERVICE_ENDPOINT,
    search_resource_id=os.getenv("AZURE_SEARCH_RESOURCE_ID", ""),
)

## Step 2: Add authentication to MCP requests

The knowledge base MCP endpoint accepts the Azure AI Search admin key that is preconfigured for this lab. Define a header provider that adds the key to each MCP request.

In [ ]:
def get_mcp_headers(_):
    return {"api-key": AZURE_SEARCH_ADMIN_KEY}

## Step 3: Create an extractive knowledge base

Create a new knowledge base over the three knowledge sources from Part 1. Minimal reasoning effort does not perform source selection, query planning, or iterative search. `EXTRACTIVE_DATA` returns retrieved evidence without attaching a model or synthesizing a knowledge base answer.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_source_names = [
    "file-docs-knowledge-source",
    "supplier-evidence-knowledge-source",
    "sourcing-documents-knowledge-source",
]

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY),
)
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova extractive knowledge base for supply-chain evidence",
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in knowledge_source_names],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Extractive knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated")

In [ ]:
verify_knowledge_base(
    index_client, KNOWLEDGE_BASE_NAME, knowledge_source_names, extractive=True
)

### Find this step in Azure portal

**Open:** Azure portal > your lab Search service > Agentic retrieval > Knowledge bases > caldova-supply-chain-minimal-knowledge-base.

**Look for:** In **Basics**, count only the three Part 1 sources. Expand **Retrieval** and **Output configurations** to inspect **Minimal** reasoning, **Extractive data**, and the empty **Chat completion model** section.

[![Original Azure portal view: Extractive knowledge base source membership.](../../img/portal/extractive-knowledge-base-sources.png)](../../img/portal/extractive-knowledge-base-sources.png)

[![Original Azure portal view: Minimal reasoning and extractive output settings.](../../img/portal/extractive-knowledge-base-settings.png)](../../img/portal/extractive-knowledge-base-settings.png)

**How to read this:** The Add model deployment button is expected here. Do not add a KB model. Search returns evidence, while the local agent will use its own model to write the answer. These are two views of the same deployed KB.

*Reference capture: 2026-10-20. Gray blocks redact environment identifiers. Orange outlines are added annotations, not status indicators. This image is not a live check of your resources.*


## Step 4: Create the knowledge base MCP tool

Every Foundry IQ knowledge base exposes a streamable HTTP MCP endpoint. Build the extractive knowledge base endpoint and expose only the `knowledge_base_retrieve` operation to the agent.

In [ ]:
knowledge_base_endpoint = (
    f"{AZURE_SEARCH_SERVICE_ENDPOINT.rstrip('/')}"
    f"/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp"
    f"?api-version={AZURE_SEARCH_API_VERSION}"
)
knowledge_base_mcp_tool = MCPStreamableHTTPTool(
    name="caldova-knowledge-base",
    url=knowledge_base_endpoint,
    header_provider=get_mcp_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
)

print(f"MCP tool configured for {knowledge_base_endpoint}")

## Step 5: Create the supply-chain agent

Create a code-first Agent Framework agent backed by the Azure OpenAI model deployed for the lab. The knowledge base returns extracts without synthesis, while the agent reasons over that evidence to produce the final answer.

In [ ]:
openai_client = AsyncOpenAI(
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=AZURE_OPENAI_KEY,
)
chat_client = OpenAIChatClient(
    model=MODEL_DEPLOYMENT_NAME,
    async_client=openai_client,
)
agent = Agent(
    client=chat_client,
    name="CaldovaSupplyChainAgent",
    instructions=(
        "You are a Caldova supply-chain assistant. "
        "Always use the Caldova knowledge base tool for supply-chain questions. "
        "Ground every answer in retrieved evidence, cite the source titles or identifiers, "
        "and keep policy, approved commitments, and operational evidence distinct. "
        "If the retrieved evidence is insufficient, say what information is unavailable."
    ),
    tools=[knowledge_base_mcp_tool],
    default_options={"store": False},
)

print(f"Agent '{agent.name}' created")

## Step 6: Run a query

Open the MCP connection for one query, then close it automatically when the response completes. Change `question` and rerun this cell to ask additional supply-chain questions without leaving an MCP lifecycle task running.

In [ ]:
question = (
    "Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. "
    "What happened, and can the 4,800 units be distributed?"
)

async with knowledge_base_mcp_tool:
    response = await agent.run(question)

display(Markdown(response.text))

## Step 7: Inspect agent activity

Agent responses include the messages produced during `run()`. Display each MCP tool call and its arguments to see the query the agent sent to the knowledge base before producing its final answer.

In [ ]:
import json

table_rows = []
for message in response.messages:
    for content in message.contents:
        if content.type == "function_call":
            arguments = content.arguments
            if not isinstance(arguments, str):
                arguments = json.dumps(arguments, indent=2)
            arguments = arguments.replace("|", "\\|").replace("\n", "<br>")
            table_rows.append((content.name, arguments))

table = ["| Call | Tool | Arguments |", "| ---: | --- | --- |"]
for call_number, (tool_name, arguments) in enumerate(table_rows, start=1):
    table.append(f"| {call_number} | `{tool_name}` | `{arguments}` |")

display(Markdown("\n".join(table)))

In [ ]:
if not any("knowledge_base_retrieve" in name for name, _ in table_rows):
    raise RuntimeError("No knowledge_base_retrieve call was recorded. This run has not verified KB grounding.")
display(Markdown(
    "**Observed:** the agent issued the tool calls above. Inspect the answer's source identifiers "
    "against the retrieved evidence. Tool activity alone does not establish that every answer claim is supported."
))

#### 🔍 Source Hunt

1. Find the `knowledge_base_retrieve` call and inspect its query.
2. Which source identifiers in the answer support the shipment excursion and distribution decision?
3. Which component synthesized the answer, and why does the KB have no attached model?
4. Why are Fabric and Work IQ absent from this agent's KB?

### Architecture checkpoint

[![Part 5 after: notebook runs the local agent, which calls its model and retrieves evidence through the MCP tool client and Search KB endpoint.](../../img/architecture/part5-after.png)](../../img/architecture/part5-after.png)

**Completed configuration:** the notebook calls `agent.run(question)` and displays `response.text`. The agent calls its model for tool selection and synthesis. It executes `knowledge_base_retrieve` through the local `MCPStreamableHTTPTool` and the Search-hosted KB endpoint. The model-free KB retrieves extracts from the three Part 1 sources. Those extracts return over the same MCP connection for the agent's answer. Solid orange shows the configured KB, its automatically exposed endpoint, tool client and local agent. No separate MCP server, hosted agent or KB model is deployed.

**Before moving on:** identify what was preconfigured, what this notebook created, and which sources actually returned evidence. The diagram is a learning guide, not a live health report.

## ✅ Mission complete

**What you built:**

* ✓ **Extractive knowledge base**: A model-free knowledge base returns evidence from the Part 1 sources without answer synthesis.
* ✓ **Authenticated MCP connection**: The agent reaches the extractive knowledge base with the lab's configured Search key.
* ✓ **Agent Framework supply-chain agent**: An Azure OpenAI-backed agent reasons over the retrieved evidence to answer an operational question.

➡️ Continue to the [lab summary](../../instructions/summary.md).